# 07 — Evaluate all registered models

This is the central comparison notebook. It discovers completed runs, validates compatible mappings, selects best checkpoints, exports common COCO JSON, computes accuracy/error/calibration metrics, and profiles models on shared hardware.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
GITHUB_USERNAME = "Harryphan72007"
GITHUB_REPOSITORY = "aerial-object-detection-benchmark"
DEFAULT_BRANCH = "main"
REPO_URL = f"https://github.com/{GITHUB_USERNAME}/{GITHUB_REPOSITORY}.git"
REPO_DIR = f"/content/{GITHUB_REPOSITORY}"
DRIVE_ROOT = "/content/drive/MyDrive/visdrone_architecture_benchmark"
assert GITHUB_USERNAME != "<MY_GITHUB_USERNAME>"
import os, sys, subprocess
if os.path.isdir(REPO_DIR) and not os.path.isdir(os.path.join(REPO_DIR, '.git')): raise RuntimeError(f'Existing non-Git directory: {REPO_DIR}')
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)
if not os.path.isdir(os.path.join(REPO_DIR, '.git')): subprocess.run(['git', 'clone', '--branch', DEFAULT_BRANCH, REPO_URL, REPO_DIR], check=True)
from src.colab_setup import clone_or_update_repository, install_project, initialize_drive_directories, load_project_config, validate_drive_writable
clone_or_update_repository(REPO_URL, REPO_DIR, DEFAULT_BRANCH)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
install_project(REPO_DIR)
config = load_project_config('project_config.yaml')
paths = initialize_drive_directories(DRIVE_ROOT)
validate_drive_writable(DRIVE_ROOT)


In [ ]:
from src.paths import ProjectPaths
from src.reproducibility import seed_everything
from src.utils.environment import collect_environment
paths = ProjectPaths.from_value(DRIVE_ROOT).create()
seed_everything(42)
collect_environment()

In [ ]:
from src.training.checkpointing import RunRegistry
import pandas as pd
registry = RunRegistry(paths)
runs = registry.list_available_runs(dataset_track="2class")
pd.DataFrame(runs)[["run_id","model_id","input_resolution","seed","best_validation_map","best_validation_aptiny"]]

In [ ]:
DATASET_TRACK="2class"
MODELS=[]  # empty means all
MAX_IMAGES=None
model_args = " ".join(MODELS)
max_arg = "" if MAX_IMAGES is None else f"--max-images {MAX_IMAGES}"
!python scripts/evaluate.py --drive-root "$DRIVE_ROOT" --dataset-track $DATASET_TRACK --best-per-model --models $model_args $max_arg!python scripts/create_results_manifest.py --drive-root "$DRIVE_ROOT" --dataset-track $DATASET_TRACK

## Efficiency profiles

Run the profiler for each selected run on the same runtime. It performs 100 warm-ups and 500 synchronized timed iterations by default. Batch-4/8 and ONNX/TensorRT are separate experiments and failures remain visible.

In [ ]:
selected = registry.list_available_runs(dataset_track=DATASET_TRACK)
for run in selected:
    print("Profile with:", f"python scripts/profile_model.py --drive-root '{DRIVE_ROOT}' --run-id {run['run_id']}")

## Resolution scaling and ablations

Use separate registered runs for 640/1024/1280 and controlled ablations. Do not resize predictions post hoc and call it a training-resolution comparison.